In [ ]:
# ACU Arthurian sources -> Google Drive
# Runs on Google's servers. Mounts your Drive, pulls each item from
# archive.org (PDF + searchable text layer), skips what is already there.
# Safe to re-run; it resumes rather than restarting.

from google.colab import drive
drive.mount('/content/drive')

DEST = '/content/drive/MyDrive/ACU_sources'   # <-- change if you want

import os, json, urllib.request, urllib.parse
os.makedirs(DEST, exist_ok=True)

ITEMS = [
 [
  "mabinogion_red_book_rhys_evans_1887",
  "textofmabinogion00rhysiala",
  "ACU-0071-0077 Welsh originals"
 ],
 [
  "cambro_british_saints_rees_1853",
  "livesofthecambro00reesuoft",
  "ACU-0007,0008,0009,0014 Latin vitae"
 ],
 [
  "percy_folio_v1_hales_furnivall_1867",
  "bishoppercysfoli01percuoft",
  "ACU-0274,0275,0277,0279"
 ],
 [
  "marie_de_france_lais_warnke_1925",
  "mariedefrancelaiswarnketroed",
  "ACU-0172,0173,0174,0179 (PDF is 257MB)"
 ],
 [
  "chronica_minora_mommsen_mgh_aa13",
  "chronicaminorasa00momm",
  "ACU-0001,0003 Latin critical"
 ],
 [
  "gildas_de_excidio_williams_1899",
  "gildaedeexcidiob00gilduoft",
  "ACU-0001 Latin+English"
 ],
 [
  "sggk_morris_eets_gollancz_1912",
  "SirGawayneAndTheGreenKnight1915",
  "ACU-0259 Middle English"
 ],
 [
  "sggk_weston_prose_1898",
  "SirGawainAndTheGreenKnightWeston1898",
  "ACU-0388 exact title match"
 ],
 [
  "scalacronica_stevenson_1836",
  "scalacronica00unkngoog",
  "ACU-0129 (mait copy has broken derivatives)"
 ],
 [
  "prose_brut_brie_eets_1906",
  "brutorchronicles00brieuoft",
  "ACU-0249 Middle English prose"
 ],
 [
  "tristan_leonis_bonilla_1912",
  "librodelesforado00boni",
  "ACU-0511 or 0515 Spanish"
 ],
 [
  "jaufre_brunel_1943_t2",
  "JaufreTomeII",
  "ACU-0213 Occitan, tome II only"
 ],
 [
  "saint_graal_hucher_1875_v1",
  "lesaintgraaloule01huchuoft",
  "ACU-0146 (Perceval text is in vol 2)"
 ],
 [
  "black_book_carmarthen_evans_1906",
  "blackbookofcarma00evanuoft",
  "ACU-0055 Welsh (weak)"
 ],
 [
  "four_ancient_books_wales_skene_v1",
  "fourancientbooks01skenuoft",
  "Skene vol 1: intro + translation"
 ],
 [
  "four_ancient_books_wales_skene_v2",
  "fourancientbooks02skenuoft",
  "ACU-0064 Welsh text (Kyuoessi, p.218ff)"
 ],
 [
  "lai_guingamor_tydorel",
  "lelaideguingamor00mari",
  "ACU-0179 Old French"
 ],
 [
  "meon_nouveau_recueil_fabliaux_1823_v1",
  "NouveauRecueilFabliauxVol1A1823",
  "ACU-0190 (contents unverified)"
 ],
 [
  "fastnachtspiele_keller_1853",
  "bub_gb_XCTIrJ7jeVUC",
  "ACU-0423 (Arthurian play NOT confirmed present)"
 ],
 [
  "layamon_brut_madden_1847_v1",
  "layamonsbrutorc00maddgoog",
  "ACU-0240"
 ],
 [
  "layamon_brut_madden_1847_v3",
  "layamonsbrutorch03layauoft",
  "ACU-0240"
 ],
 [
  "holinshed_chronicles_ellis_1807_v1",
  "holinshedschroni01holi",
  "ACU-0305"
 ],
 [
  "portugaliae_mon_hist_scriptores_v1",
  "portugaliaemonumentahistoricascrv1",
  "ACU-0519 (unverified)"
 ],
 [
  "demanda_sancto_grial_spanish_print",
  "primeroysegundolibrodegria",
  "Spanish Demanda, distinct from ACU-0506"
 ]
]

UA = {'User-Agent': 'ACU-acquisition/1.0'}
def get(url, timeout=300):
    return urllib.request.urlopen(urllib.request.Request(url, headers=UA), timeout=timeout)

ok = skip = fail = 0
for slug, ident, note in ITEMS:
    d = os.path.join(DEST, slug); os.makedirs(d, exist_ok=True)
    print('==', slug, '|', note)
    try:
        md = json.load(get('https://archive.org/metadata/' + ident))
    except Exception as e:
        print('   metadata failed:', e); fail += 1; continue
    want = []
    for f in md.get('files', []):
        if f.get('format') in ('Text PDF','Image Container PDF') and not any(
                n['name'].endswith('.pdf') for n in want):
            want.append(f)
        if f.get('format') == 'DjVuTXT':
            want.append(f)
    if not want:
        print('   no pdf/text derivative'); fail += 1; continue
    for f in want:
        name = f['name']; path = os.path.join(d, name)
        size = int(f.get('size', 0) or 0)
        if os.path.exists(path) and os.path.getsize(path) == size:
            print('   have', name); skip += 1; continue
        url = 'https://archive.org/download/%s/%s' % (ident, urllib.parse.quote(name))
        try:
            with get(url) as r, open(path, 'wb') as w:
                while True:
                    c = r.read(1 << 20)
                    if not c: break
                    w.write(c)
            print('   got ', name, '%.1f MB' % (os.path.getsize(path)/1e6)); ok += 1
        except Exception as e:
            print('   FAILED', name, e); fail += 1
    open(os.path.join(d, '_source.txt'), 'w').write('%s\t%s\n' % (ident, note))

print('\ndownloaded %d | already had %d | failed %d' % (ok, skip, fail))
print('location:', DEST)
